In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

In [ ]:
df=pd.read_csv("../data/processed/daily_sales_processed.csv")
df.head()

In [ ]:
#Create X and y for model training and prediction
X=df[['Year','Month','Day','Weekday','Lag_1','Lag_7','Lag_30','Rolling_7_mean','Rolling_30_mean']] #Double brackets create a dataframe
y=df['Sales'] #Single bracket creates a series
X.head()
y.head()
X.shape
y.shape

In [ ]:
#We want to use the old data for training and the newer data for testing.
#Hence we don't use train_test_split and will split the data manually
train_size=int(len(df)*0.8)
X_train=X[:train_size]
X_test=X[train_size:]
y_train=y[:train_size]
y_test=y[train_size:]
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

In [ ]:
#First we will use a simple model like Linear Regression which is basic industry-level and see if we need to improve the model
from sklearn.linear_model import LinearRegression
model=LinearRegression()
model.fit(X_train,y_train)
model.coef_
model.intercept_
y_pred=model.predict(X_test)
print(y_test.head())
print(y_pred[:5])#Helps us understand actual vs predicted values which is way off here suggesting model isn't good for this case.

#Running evaluation metrics for better understanding whether model is suitable or not.
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
mae=mean_absolute_error(y_test,y_pred)
mse=mean_squared_error(y_test,y_pred)
rmse=np.sqrt(mse)
r2=r2_score(y_test,y_pred)
print("MAE : " + str(mae))
print("RMSE : " + str(rmse))
print("R2 : " + str(r2)) #R2 close to 0 means it predicts only avreage results and does not undertand and predict spikes, basic prediction through only guessing average and not understanding variations like seasonality etc.

In [ ]:
#Comparing actual vs predicted sales values for better understanding
plt.figure(figsize=(12,6))

plt.plot(y_test.values, label='Actual Sales')
plt.plot(y_pred, label='Predicted Sales')

plt.legend()
plt.title("Actual vs Predicted Sales")

plt.show()

In [ ]:
#Implementing our model
from sklearn.ensemble import RandomForestRegressor
rf_model=RandomForestRegressor(
    n_estimators=100,
    random_state=42
)
rf_model.fit(X_train,y_train)


In [ ]:
#Predicting output and Evaluating model
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
y_pred_rf=rf_model.predict(X_test)
print(y_pred_rf[:5])
print(y_test[:5])#Comparing initial prediction and test values

#Evaluating model
rf_mae = mean_absolute_error(y_test, y_pred_rf)

rf_rmse = np.sqrt(
    mean_squared_error(y_test, y_pred_rf)
)
rf_r2 = r2_score(y_test, y_pred_rf)
print("MAE :", rf_mae)
print("RMSE :", rf_rmse)
print("R2 :", rf_r2)

In [ ]:
#We are getting high errors and really off values in the prediction. Hence we need to inspect our data as well as our model.
#Data inspection is to be done first and for it we need to inspect feature importance.
feature_importance=pd.DataFrame({
    'Feature' : X.columns,
    'Importance' : rf_model.feature_importances_
})
feature_importance.sort_values(
    by='Importance',
    ascending=False
)

#We have added Lag_7 and Lag_30 to see if error reduces and predictions get stronger.
#However still our errors are high and predictions are off. Before trying to change the model lets try to optimize the hyperparameters using GridSearchCV.

In [ ]:
#Final model-Got decent accuracy not improving more at current stage.
rom sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
param_grid={
    'n_estimators' : [100,200,300],
    'max_depth' : [5,10,15,None],
    'min_samples_split' : [2,5,10]
}
grid=GridSearchCV(rf_model,param_grid=param_grid,cv=3,scoring='neg_mean_absolute_error',n_jobs=-1)
grid.fit(X_train,y_train)
print(grid.best_params_)
best_rf=grid.best_estimator_
y_pred=best_rf.predict(X_test)
print(y_test[:5])
print(y_pred[:5])

print("MAE :", mean_absolute_error(y_test, y_pred))
print("RMSE :", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 :", r2_score(y_test, y_pred))